# Two-tower & neural retrieval recommenders — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x)))
def dcg(rels): return sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rels))
def ndcg(rels):
    best=sorted(rels, reverse=True)
    return dcg(rels)/dcg(best) if dcg(best)>0 else 0.0
print('setup ok')

## Embedding users and items so nearest-neighbor search retrieves candidates
Two-tower retrieval scores a user embedding against many item embeddings. The examples compute dot-product retrieval, temperature-softmax training loss, in-batch negatives, approximate top-k by projection, and a small recall@k check.

In [ ]:
# 1) dot-product retrieval ranks item vectors for one user vector
u=np.array([1.0,0.2]); V=np.array([[1,.1],[.8,.4],[.1,1],[-.5,.8]],float)
scores=V@u
plt.figure(figsize=(5,4)); plt.scatter(V[:,0],V[:,1]); plt.quiver([0],[0],[u[0]],[u[1]],angles='xy',scale_units='xy',scale=1,color='r'); plt.title(f'top item = {int(np.argmax(scores))}')
assert int(np.argmax(scores))==0 and abs(scores[0]-1.02)<1e-12

In [ ]:
# 2) softmax retrieval loss compares the positive to all sampled negatives
scores=np.array([2.0,1.0,0.0]); p=np.exp(scores)/np.exp(scores).sum(); loss=float(-np.log(p[0]))
plt.figure(figsize=(6,3)); plt.bar(['pos','neg1','neg2'],p); plt.title(f'positive prob={p[0]:.3f}, loss={loss:.3f}')
assert abs(loss-0.40760596444438046)<1e-12

In [ ]:
# 3) temperature controls how sharp the retrieval distribution is
scores=np.array([2.0,1.0,0.0]); temps=[.5,1,2]; pos=[]
for t in temps:
    p=np.exp(scores/t)/np.exp(scores/t).sum(); pos.append(p[0])
plt.figure(figsize=(6,3)); plt.bar([str(t) for t in temps],pos); plt.xlabel('temperature'); plt.ylabel('P(positive)'); plt.title('lower temperature sharpens')
assert pos[0]>pos[1]>pos[2]

In [ ]:
# 4) in-batch negatives use every other row's item as a negative
U=np.array([[1,.1],[.1,1],[.8,.2]],float); V=np.array([[1,0],[0,1],[.7,.3]],float); S=U@V.T
plt.figure(figsize=(4,4)); plt.imshow(S,cmap='magma'); plt.colorbar(label='dot'); plt.title('batch score matrix')
assert S.shape==(3,3) and S[0,0]>S[0,1]

In [ ]:
# 5) recall@k checks whether the true item appears in retrieved candidates
S=np.array([[1.0,.1,.8],[.2,.9,.3],[.4,.5,.7]]); true=np.array([0,1,2]); k=2
top=np.argsort(-S,axis=1)[:,:k]; recall=float(np.mean([true[i] in top[i] for i in range(3)]))
plt.figure(figsize=(6,3)); plt.bar(['q0','q1','q2'],[true[i] in top[i] for i in range(3)]); plt.title(f'recall@2 = {recall:.3f}')
assert abs(recall-1.0)<1e-12